Machine Learning / Advanced Analysis

# # Response‑Time Regression: Predict Student Answer Times
# This notebook walks through a step‑by‑step pipeline to predict how long a student will take to answer a question using the EdNet dataset.
# **Goal:** Build regression models (Random Forest, XGBoost, and MLP) to predict response time.
**Approach:**
1. Load and merge cleaned KT1–KT4 data.
2. Feature engineering: past average time per part, time‑of‑day, question difficulty.
3. Train/test split stratified by question part.
4. Model training with pipelines.
5. Evaluation using RMSE and MAE.

In [3]:
# 1. Imports and Setup
import pandas as pd
import numpy as np
from datetime import datetime
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.layers import Input, Embedding, Dense, Flatten, Concatenate, Dropout
from tensorflow.keras.models import Model
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error
import warnings
warnings.filterwarnings('ignore')


In [7]:
# 2. Load and Merge Data
data = pd.read_parquet('EdNet_full_log.parquet')
print("Data shape:", data.shape)

# Merge in question metadata (bundle_id and part) from questions.csv
questions = pd.read_csv('questions.csv', usecols=['question_id', 'bundle_id', 'part'])
# Ensure matching types
# If question_id in data is string prefixed with 'q', strip; otherwise, cast as needed
# questions['question_id'] = questions['question_id'].str.lstrip('q')  # uncomment if needed
# data['question_id'] = data['question_id'].astype(str)

data = data.merge(questions, on='question_id', how='left')
print("After merge shape:", data.shape)
print(data[['question_id','bundle_id','part']].drop_duplicates().head())

Data shape: (32518836, 13)
After merge shape: (32518836, 15)
   question_id bundle_id  part
0         None       NaN   NaN
6        q5012     b3544   5.0
16       q4706     b3238   5.0
26       q4366     b2898   5.0
36       q4829     b3361   5.0


In [8]:
# 3. Feature Engineering

## 3.1 Past Average Response Time per Part
# Compute per-student, per-part average elapsed_time
avg_time = (
    data.groupby(['user_id', 'part'])['elapsed_time']
        .mean()
        .reset_index()
        .rename(columns={'elapsed_time':'avg_time_per_part'})
)
# Merge back
data = data.merge(avg_time, on=['user_id','part'], how='left')

## 3.2 Time‑of‑Day and Categorical Encoding
# Extract hour and bin into time_of_day
data['hour'] = data['timestamp'].dt.hour
bins = [0,6,12,18,24]
labels = ['night','morning','afternoon','evening']
data['time_of_day'] = pd.cut(data['hour'], bins=bins, labels=labels, right=False)

# Encode time_of_day and bundle_id for embeddings
le_tod = LabelEncoder()
data['tod_idx'] = le_tod.fit_transform(data['time_of_day'].astype(str))
le_bundle = LabelEncoder()
data['bundle_idx'] = le_bundle.fit_transform(data['bundle_id'].astype(str))

## 3.3 Final Feature Matrix and Target
features = ['avg_time_per_part','tod_idx','bundle_idx']
target = 'elapsed_time'
X = data[features]
y = data[target]

In [12]:
#4. Handle Missing Values
# Before splitting, drop any rows with missing target or feature values
# This avoids NaNs in y causing errors
mask = data[['elapsed_time','avg_time_per_part','tod_idx','bundle_idx','part']].notnull().all(axis=1)
data = data.loc[mask].reset_index(drop=True)

# Recompute feature matrix and target to match filtered data (or else X wont match Y)
X = data[features]
y = data[target]


In [13]:
#5. Train/Test Split
# Stratify by question part to maintain distribution

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    random_state=42,
    stratify=data['part']
)

# 5. Traditional Models (Random Forest & XGBoost)

In [15]:
# 5.1 Preprocessing Pipeline
numeric_features = ['avg_time_per_part']
categorical_features = ['tod_idx','bundle_idx']

numeric_transformer = Pipeline([('scaler', StandardScaler())])
categorical_transformer = Pipeline([('onehot', OneHotEncoder(handle_unknown='ignore'))])

preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

In [16]:
# 5.2 Define Pipelines
pipeline_rf = Pipeline([('preproc', preprocessor),('rf', RandomForestRegressor(random_state=42))])
pipeline_xgb = Pipeline([('preproc', preprocessor),('xgb', XGBRegressor(objective='reg:squarederror', random_state=42))])

In [ ]:
# 5.3 Hyperparameter Tuning for Random Forest
param_grid_rf = {
    'rf__n_estimators': [100, 200],
    'rf__max_depth': [None, 10, 20]
}
grid_rf = GridSearchCV(
    pipeline_rf,
    param_grid_rf,
    cv=3,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1
)
grid_rf.fit(X_train, y_train)
print("Best RF params:", grid_rf.best_params_)

In [ ]:
# 5.4 Evaluate RF and XGBoost

# Random Forest evaluation
best_rf = grid_rf.best_estimator_
rf_pred = best_rf.predict(X_test)
rf_rmse = mean_squared_error(y_test, rf_pred, squared=False)
rf_mae = mean_absolute_error(y_test, rf_pred)

# XGBoost evaluation
pipeline_xgb.fit(X_train, y_train)
xgb_pred = pipeline_xgb.predict(X_test)
xgb_rmse = mean_squared_error(y_test, xgb_pred, squared=False)
xgb_mae = mean_absolute_error(y_test, xgb_pred)

print(f"RF RMSE: {rf_rmse:.2f}, MAE: {rf_mae:.2f}")
print(f"XGB RMSE: {xgb_rmse:.2f}, MAE: {xgb_mae:.2f}")

KeyboardInterrupt: 